# 开发集与冻结保留集评价

指标以 query 宏平均，区间以共享 gold／近重复组件配对 bootstrap。开发区间是探索证据；保留集仅接受冻结模型、候选和本地预先记录协议的评价，不据其结果继续调参。语义审阅为 AI 辅助，实际成员尚未独立核验。

In [1]:
from pathlib import Path
import json, sys, os
import pandas as pd
from IPython.display import display, Image
root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'project/src/ragstudy').is_dir())
for folder in ('.vendor', '.vendor-hf', '.vendor-onnx-compat'):
    path = root / 'project' / folder
    if path.is_dir():
        sys.path.insert(0, str(path))
sys.path.insert(0, str(root / 'project/src'))
artifacts = sorted(path for path in (root / 'project/artifacts').iterdir()
                   if (path / 'results.json').is_file() and (path / 'search_bundle.joblib').is_file())
assert artifacts, 'First run python project/scripts/run_experiment.py --backend lsa'
artifact = Path(os.environ['RAG_ARTIFACT']).resolve() if os.environ.get('RAG_ARTIFACT') else artifacts[-1]
result = json.loads((artifact / 'results.json').read_text(encoding='utf-8'))
print('Actual local run:', artifact.name, '| backend:', result['retrieval_metadata']['semantic']['method'])

Actual local run: frozen_e2324b7a967c5dbe2766ecc3 | backend: frozen_onnx_fp32_cls_cosine


In [2]:
display(pd.DataFrame(result['summary']['dev']['methods']).T)
display(pd.DataFrame(result['summary']['dev']['paired_comparisons']).T[['difference','ci_lower','ci_upper','n_queries','n_groups']])
print('Final test evaluated:', result['final_test_evaluated'])
if result['final_test_evaluated']:
    assert result['frozen_evaluation']['no_new_training_or_indexing'] is True
    display(pd.DataFrame(result['summary']['test']['methods']).T)
    display(result['primary_holdout_contrast'])
else:
    assert 'test' not in result['summary']

,ndcg@10,recall@20
bm25,0.599669,0.690476
semantic,0.686528,0.797619
rrf,0.656692,0.797619
weighted_rrf,0.667284,0.833333
logistic,0.669473,0.773810
random_forest,0.680588,0.821429


,difference,ci_lower,ci_upper,n_queries,n_groups
logistic,0.012781,-0.003187,0.031535,84,72
random_forest,0.023897,0.002658,0.051765,84,72


Final test evaluated: True


,ndcg@10,recall@20
bm25,0.510022,0.654762
semantic,0.577842,0.750000
rrf,0.553764,0.726190
weighted_rrf,0.584744,0.726190
logistic,0.578763,0.738095
random_forest,0.590379,0.738095


{'baseline': 'weighted_rrf',
 'challenger': 'logistic',
 'metric': 'ndcg@10',
 'baseline_mean': 0.5847443390803866,
 'challenger_mean': 0.578763264594831,
 'difference': -0.005981074485555626,
 'ci_lower': -0.023426474624533213,
 'ci_upper': 0.011785201428518877,
 'confidence_level': 0.95,
 'resampling_unit': 'group',
 'aggregation_unit': 'query',
 'n_queries': 84,
 'n_groups': 72,
 'iterations': 2000,
 'seed': 20261114}

In [3]:
verification_path = artifact / 'verification.json'
if verification_path.is_file():
    verification = json.loads(verification_path.read_text(encoding='utf-8'))
    display(verification)
else:
    print('Run scripts/verify_experiment.py for independent SQL/metric/candidate verification.')

{'status': 'passed',
 'run_id': 'frozen_e2324b7a967c5dbe2766ecc3',
 'verified_at_utc': '2026-11-14T14:58:26.557187+00:00',
 'artifact_directory': 'D:\\Users\\hyy\\Desktop\\guideline\\course\\5001\\project\\artifacts\\frozen_e2324b7a967c5dbe2766ecc3',
 'checks': {'exact_candidate_union': 'saved_retrieval_runs',
  'tracked_csv': 'passed',
  'configuration_and_input_hashes': 'passed',
  'sqlite_integrity': 'passed',
  'component_and_shared_gold_split_isolation': 'passed',
  'complete_evaluation_query_cohort': 'passed',
  'candidate_label_consistency': 'passed',
  'independent_ranking_metrics': 'passed',
  'independent_aggregate_metrics': 'passed',
  'same_complete_candidate_set_membership': 'passed',
  'default_test_ranking_lock': 'passed',
  'bootstrap_scope': 'means, group counts and interval bounds verified; interval draws not independently recomputed'},
 'warnings': [],
 'validation_source_sha256': {'project/scripts/run_experiment.py': 'c30c1c7119a5392a35e35d4dae9597d1f148e10c8ff34b34

In [4]:
latency_path = artifact / 'latency_benchmark.json'
if latency_path.is_file():
    latency = json.loads(latency_path.read_text(encoding='utf-8'))
    display(pd.DataFrame({name: values['total'] for name,values in latency['summary'].items()}).T)
    print(latency['timing_boundary'])
else:
    print('Repeated warm latency has not been measured for this run.')

Repeated warm latency has not been measured for this run.
